# 비용 청구 분석

이 노트북은 플러그인을 사용하여 로컬 영수증 이미지에서 여행 경비를 처리하고, 비용 청구 이메일을 생성하며, 파이 차트를 사용해 경비 데이터를 시각화하는 에이전트를 만드는 방법을 보여줍니다. 에이전트는 작업 상황에 따라 동적으로 함수를 선택합니다.

단계:
1. OCR 에이전트는 로컬 영수증 이미지를 처리하고 여행 경비 데이터를 추출합니다.
2. 이메일 에이전트는 비용 청구 이메일을 생성합니다.

### 여행 경비 시나리오 예시:
직원으로서 다른 도시에 출장 가는 상황을 상상해 보십시오. 회사는 합리적인 모든 여행 관련 경비를 환급해 주는 정책을 가지고 있습니다. 가능한 여행 경비 내역은 다음과 같습니다:
- 교통:
귀하의 출발 도시에서 목적지 도시까지 왕복 항공권.
공항으로 가고 오는 택시 또는 차량 호출 서비스.
목적지 도시 내 현지 교통수단 (예: 대중교통, 렌터카, 택시 등).

- 숙박:
회의 장소 근처 중급 비즈니스 호텔에서 3박 숙박.

- 식사:
회사 일일 식대 정책에 따른 아침, 점심, 저녁의 일일 식대.

- 기타 경비:
공항 주차 요금.
호텔 인터넷 접속 요금.
팁이나 작은 서비스 요금.

- 문서화:
환급을 위해 모든 영수증(항공권, 택시, 호텔, 식사 등)과 작성된 비용 보고서를 제출합니다.


## 필요한 라이브러리 가져오기

노트북에 필요한 라이브러리와 모듈을 가져옵니다.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## 비용 모델 정의

개별 비용에 대한 Pydantic 모델과 사용자 쿼리를 구조화된 비용 데이터로 변환하는 ExpenseFormatter 클래스를 만듭니다.

각 비용은 다음 형식으로 표현됩니다:
`{'date': '07-Mar-2025', 'description': '목적지로 가는 비행기', 'amount': 675.99, 'category': '교통'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## 도구 정의 - 이메일 생성

비용 청구서를 제출하기 위한 이메일을 생성하는 도구 함수를 만드세요.  
- 이 도구는 Microsoft Agent Framework의 `@tool` 데코레이터를 사용합니다.  
- 비용의 총 금액을 계산하고 세부 정보를 이메일 본문 형식으로 만듭니다.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# 영수증 이미지에서 여행 경비를 추출하는 도구

영수증 이미지에서 여행 경비를 추출하는 도구 함수를 만드세요.
- 이 도구는 Microsoft Agent Framework의 `@tool` 데코레이터를 사용합니다.
- 영수증 이미지를 읽고 base64로 인코딩한 후, 에이전트가 분석할 수 있도록 데이터 URI를 반환합니다.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## 비용 처리

에이전트를 정의하고 `WorkflowBuilder`를 사용하여 순차적 워크플로우에 연결합니다.
- OCR 에이전트는 `load_receipt_image` 도구를 사용하여 영수증 이미지에서 구조화된 비용 데이터를 추출합니다.
- 이메일 에이전트는 추출된 데이터를 받아 `generate_expense_email` 도구를 사용하여 전문적인 비용 청구 이메일을 생성합니다.
- `add_edge`가 포함된 `WorkflowBuilder`는 순차 파이프라인을 생성합니다: OCR 에이전트 → 이메일 에이전트.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Main function

순차적 워크플로우를 구축하고 실행하여 영수증 이미지를 처리하고 비용 청구 이메일을 생성합니다.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:  
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 위해 노력하고 있으나, 자동 번역에는 오류나 부정확함이 포함될 수 있음을 유의하시기 바랍니다. 원본 문서는 해당 언어의 원문이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우 전문적인 인간 번역을 권장합니다. 본 번역의 사용으로 인해 발생하는 어떠한 오해나 잘못된 해석에 대해서도 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
